# Contrastive Autoencoder for Emergent Representation Learning

**v2 — Kaggle/Colab compatible**

Key changes from v1:
- Removed VAE (KL fighting against structure)
- Added contrastive loss (push similar sequences together)
- Smaller latent (128 dims)
- Synthetic data with known structure first

---

## Goal
Discover self-organised latent structures without human-imposed semantics — but using a method that actually enables this.

---

## Contents
1. Environment setup
2. Data generation (synthetic with known structure)
3. Contrastive autoencoder model
4. Training with reconstruction + contrastive loss
5. Latent space analysis
6. Clustering & visualisation
7. Comparison: synthetic patterns discovered vs known


## Cell 1 — Detect runtime environment

In [ ]:
import sys, os, subprocess

IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ

PLATFORM = "Kaggle" if IN_KAGGLE else ("Colab" if IN_COLAB else "Local/Other")
print(f"Platform: {PLATFORM}")

# GPU check
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"GPU: {result.stdout.strip()}")
else:
    print("GPU: not detected")

## Cell 2 — Install dependencies

In [ ]:
import subprocess, sys

def install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

packages = ["torch", "numpy", "scikit-learn", "matplotlib", "seaborn", "umap-learn", "tqdm"]
for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
    except:
        install([pkg])
print("Dependencies ready")

## Cell 3 — Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Cell 4 — DNA Encoder / Decoder

In [ ]:
def encode_dna(sequences, window_size=1024):
    """One-hot encode DNA sequences."""
    encoding = {"A": [1,0,0,0], "C": [0,1,0,0], "G": [0,0,1,0], "T": [0,0,0,1], "N": [0,0,0,0]}
    encoded = []
    for seq in sequences:
        seq = seq.upper()[:window_size]
        seq = seq.ljust(window_size, "N")
        encoded.append([encoding.get(b, [0,0,0,0]) for b in seq])
    return np.array(encoded, dtype=np.float32).reshape(-1, window_size * 4)

def decode_dna(encoded, window_size=1024):
    """Decode one-hot back to sequence."""
    bases = ["A", "C", "G", "T", "N"]
    sequences = []
    for enc in encoded:
        enc = enc.reshape(window_size, 4)
        seq = ""
        for row in enc:
            idx = np.argmax(row)
            seq += bases[idx] if idx < 4 else "N"
        sequences.append(seq)
    return sequences

def gc_content(sequences):
    """Calculate GC content of sequences."""
    gc = []
    for seq in sequences:
        seq = seq.upper()
        g = seq.count("G")
        c = seq.count("C")
        gc.append((g + c) / len(seq) * 100 if len(seq) > 0 else 0)
    return np.array(gc)

print("DNA encoder/decoder ready")

## Cell 5 — Synthetic Data Generator with Known Structure

In [ ]:
def generate_synthetic_genome(n_sequences=100000, window_size=1024, n_patterns=5, seed=42):
    """
    Generate synthetic DNA with known patterns.
    Each pattern has a specific GC content and k-mer bias.
    """
    np.random.seed(seed)
    
    # Define known patterns with different GC content
    patterns = []
    for i in range(n_patterns):
        # Different GC content for each pattern
        gc = 0.2 + (i / (n_patterns - 1)) * 0.6  # 20% to 80%
        
        # Generate base frequencies
        freq = {
            "A": (1 - gc) / 2,
            "T": (1 - gc) / 2,
            "G": gc / 2,
            "C": gc / 2
        }
        
        # Add k-mer bias (known structure)
        bias_kmer = "ATG" + "".join(["G" if j % 2 == 0 else "C" for j in range(5)])[:5]
        
        # Generate sequences for this pattern
        for _ in range(n_sequences // n_patterns):
            seq = ""
            for _ in range(window_size):
                r = np.random.random()
                if r < freq["A"]:
                    seq += "A"
                elif r < freq["A"] + freq["T"]:
                    seq += "T"
                elif r < freq["A"] + freq["T"] + freq["G"]:
                    seq += "G"
                else:
                    seq += "C"
            # Inject k-mer bias
            pos = np.random.randint(0, window_size - len(bias_kmer))
            seq = seq[:pos] + bias_kmer + seq[pos + len(bias_kmer):]
            patterns.append(seq)
    
    random.shuffle(patterns)
    return patterns[:n_sequences]

# Generate training data
print("Generating synthetic data with known structure...")
n_sequences = 10000
window_size = 1024
n_patterns = 5

synthetic_seqs = generate_synthetic_genome(n_sequences, window_size, n_patterns)
X_synthetic = encode_dna(synthetic_seqs, window_size)

print(f"Generated {len(synthetic_seqs)} sequences")
print(f"Shape: {X_synthetic.shape}")
print(f"GC content per pattern group:")
for i in range(n_patterns):
    start = i * (len(synthetic_seqs) // n_patterns)
    end = (i + 1) * (len(synthetic_seqs) // n_patterns)
    gc = np.mean(gc_content(synthetic_seqs[start:end]))
    print(f"  Pattern {i}: {gc:.1f}% GC")

## Cell 6 — Dataset with Augmentation

In [ ]:
class GenomicDataset(Dataset):
    def __init__(self, encoded_sequences, window_size=1024, augment_prob=0.3, noise_scale=0.05):
        self.data = torch.from_numpy(encoded_sequences).float()
        self.window_size = window_size
        self.augment_prob = augment_prob
        self.noise_scale = noise_scale

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        if random.random() < self.augment_prob:
            noise = torch.randn_like(x) * self.noise_scale
            x = torch.clamp(x + noise, 0.0, 1.0)
        return x, x


class ContrastiveDataset(Dataset):
    """Dataset that returns anchor / positive / negative samples for contrastive loss."""
    def __init__(self, encoded_sequences, window_size=1024, k_neighbors=5):
        self.data = torch.from_numpy(encoded_sequences).float()
        self.window_size = window_size
        self.k = k_neighbors

        print("Computing similarity neighbours...")
        from sklearn.neighbors import NearestNeighbors
        self.neigh = NearestNeighbors(
            n_neighbors=min(len(self.data), self.k + 5),
            metric="cosine",
            n_jobs=-1
        )
        self.neigh.fit(self.data.numpy())
        self.distances, self.indices = self.neigh.kneighbors(self.data.numpy())

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]

        neighbors = self.indices[idx]
        positive_candidates = neighbors[neighbors != idx]
        if len(positive_candidates) == 0:
            positive_idx = idx
        else:
            positive_idx = int(np.random.choice(positive_candidates))

        negative_idx = idx
        while negative_idx == idx or negative_idx in positive_candidates:
            negative_idx = np.random.randint(0, len(self.data))

        x_positive = self.data[positive_idx]
        x_negative = self.data[negative_idx]

        return x, x_positive, x_negative


# Create datasets
train_size = int(0.9 * len(X_synthetic))
X_train, X_val = X_synthetic[:train_size], X_synthetic[train_size:]

train_dataset = ContrastiveDataset(X_train, window_size)
val_dataset = GenomicDataset(X_val, window_size)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")


## Cell 7 — Contrastive Autoencoder Model

In [ ]:
class ContrastiveAutoencoder(nn.Module):
    def __init__(self, input_dim=4096, latent_dim=128, dropout=0.3):
        super().__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 2048),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, latent_dim)
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 2048),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(2048, input_dim),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

# Create model
model = ContrastiveAutoencoder(input_dim=window_size*4, latent_dim=128, dropout=0.3).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## Cell 8 — Loss Functions

In [ ]:
class NT_XentLoss(nn.Module):
    """Normalized Temperature-scaled Cross Entropy Loss for contrastive learning."""
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, z1, z2):
        # z1 and z2 are (batch_size, latent_dim)
        # Normalize
        z1 = nn.functional.normalize(z1, dim=1)
        z2 = nn.functional.normalize(z2, dim=1)
        
        # Similarity
        sim = torch.matmul(z1, z2.T) / self.temperature
        
        # Loss
        labels = torch.arange(len(z1), device=z1.device)
        loss = nn.functional.cross_entropy(sim, labels)
        
        return loss

# Reconstruction loss
criterion_mse = nn.MSELoss()
criterion_contrastive = NT_XentLoss(temperature=0.1)

def total_loss(x_recon, x_target, z_anchor, z_positive, lambda_contrastive=0.1):
    """Combined reconstruction + contrastive loss."""
    recon_loss = criterion_mse(x_recon, x_target)
    contrastive_loss = criterion_contrastive(z_anchor, z_positive)
    return recon_loss + lambda_contrastive * contrastive_loss

print("Loss functions ready")

## Cell 9 — Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, lambda_contrastive=0.1):
    model.train()
    total_recon = 0
    total_contrastive = 0
    
    for x, x_pos, x_neg in tqdm(loader, desc="Training"):
        x = x.to(device)
        x_pos = x_pos.to(device)
        x_neg = x_neg.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        x_recon, z = model(x)
        _, z_pos = model(x_pos)
        
        # Reconstruction loss
        recon_loss = criterion_mse(x_recon, x)
        
        # Contrastive loss (anchor-positive pair)
        contrastive_loss = criterion_contrastive(z, z_pos)
        
        # Total loss
        loss = recon_loss + lambda_contrastive * contrastive_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_recon += recon_loss.item()
        total_contrastive += contrastive_loss.item()
    
    return total_recon / len(loader), total_contrastive / len(loader)

def validate(model, loader):
    model.eval()
    total_recon = 0
    
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            x_recon, _ = model(x)
            recon_loss = criterion_mse(x_recon, x)
            total_recon += recon_loss.item()
    
    return total_recon / len(loader)

print("Training functions ready")

## Cell 10 — Execute Training

In [ ]:
# Training config
epochs = 30
lr = 1e-3
lambda_contrastive = 0.1

optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

history = {"train_recon": [], "train_contrastive": [], "val_recon": []}

best_val_loss = float('inf')

for epoch in range(epochs):
    train_recon, train_contrast = train_epoch(model, train_loader, optimizer, lambda_contrastive)
    val_recon = validate(model, val_loader)
    scheduler.step()
    
    history["train_recon"].append(train_recon)
    history["train_contrastive"].append(train_contrast)
    history["val_recon"].append(val_recon)
    
    if val_recon < best_val_loss:
        best_val_loss = val_recon
        torch.save(model.state_dict(), "best_model.pth")
    
    print(f"Epoch {epoch+1}/{epochs} | Train Recon: {train_recon:.4f} | Train Contrastive: {train_contrast:.4f} | Val Recon: {val_recon:.4f}")

print("Training complete!")

## Cell 11 — Training Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history["train_recon"]) + 1)

axes[0].plot(epochs_range, history["train_recon"], label="Train Recon")
axes[0].plot(epochs_range, history["val_recon"], label="Val Recon")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Reconstruction Loss (MSE)")
axes[0].set_title("Reconstruction Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, history["train_contrastive"])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Contrastive Loss")
axes[1].set_title("Contrastive Loss")
axes[1].grid(True)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

print("Training curves saved")

## Cell 12 — Latent Space Extraction

In [ ]:
# Load best model
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# Extract latents
all_latents = []
all_labels = []

with torch.no_grad():
    for i in range(0, len(X_synthetic), batch_size):
        batch = torch.FloatTensor(X_synthetic[i:i+batch_size]).to(device)
        z = model.encode(batch)
        all_latents.append(z.cpu().numpy())
        
        # Label by original pattern group
        label = i // (len(X_synthetic) // n_patterns)
        all_labels.extend([label] * len(batch))

latents = np.vstack(all_latents)
labels = np.array(all_labels[:len(latents)])

print(f"Latent shape: {latents.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Unique labels: {np.unique(labels)}")

## Cell 13 — UMAP Visualisation

In [ ]:
# UMAP projection
print("Computing UMAP...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED)
latents_umap = reducer.fit_transform(latents)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(latents_umap[:, 0], latents_umap[:, 1], c=labels, cmap="tab10", alpha=0.5, s=5)
plt.colorbar(scatter, label="Pattern Group")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("Latent Space (UMAP) - Colored by Known Pattern")
plt.savefig("latent_umap.png", dpi=150)
plt.show()

print("UMAP saved")

## Cell 14 — Clustering Analysis

In [ ]:
# K-means clustering
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    cluster_labels = kmeans.fit_predict(latents)
    score = silhouette_score(latents, cluster_labels)
    silhouette_scores.append(score)
    print(f"k={k}: silhouette={score:.4f}")

best_k = k_range[np.argmax(silhouette_scores)]
print(f"\nBest k: {best_k}")

# Visualise best clustering
kmeans_best = KMeans(n_clusters=best_k, random_state=SEED, n_init=10)
cluster_labels = kmeans_best.fit_predict(latents)

plt.figure(figsize=(10, 8))
plt.scatter(latents_umap[:, 0], latents_umap[:, 1], c=cluster_labels, cmap="tab10", alpha=0.5, s=5)
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title(f"Discovered Clusters (k={best_k})")
plt.savefig("clustering_analysis.png", dpi=150)
plt.show()

## Cell 15 — Pattern Discovery Comparison

In [ ]:
# Compare discovered clusters vs known patterns
print("=== Pattern Discovery Analysis ===\n")

print("Known pattern GC content:")
known_gc = []
for i in range(n_patterns):
    start = i * (len(synthetic_seqs) // n_patterns)
    end = (i + 1) * (len(synthetic_seqs) // n_patterns)
    gc = np.mean(gc_content(synthetic_seqs[start:end]))
    known_gc.append(gc)
    print(f"  Pattern {i}: {gc:.1f}% GC")

print("\nCluster GC content:")
for i in range(best_k):
    cluster_idx = np.where(cluster_labels == i)[0]
    # Get sequences for this cluster
    cluster_seqs = [synthetic_seqs[j] for j in cluster_idx[:1000]]
    cluster_gc = np.mean(gc_content(cluster_seqs))
    print(f"  Cluster {i}: {cluster_gc:.1f}% GC")

print("\n=== Interpretation ===")
print("If clusters match known patterns by GC content, the model discovered the structure!")

## Cell 16 — Reconstruction Quality

In [ ]:
# Test reconstruction
with torch.no_grad():
    sample = torch.FloatTensor(X_val[:100]).to(device)
    recon, _ = model(sample)
    recon = recon.cpu().numpy()

recon_seqs = decode_dna(recon, window_size)
original_seqs = decode_dna(X_val[:100], window_size)

original_gc = gc_content(original_seqs)
recon_gc = gc_content(recon_seqs)

print("Reconstruction GC content:")
print(f"  Original: {np.mean(original_gc):.1f}% ± {np.std(original_gc):.1f}%")
print(f"  Reconstructed: {np.mean(recon_gc):.1f}% ± {np.std(recon_gc):.1f}%")

mse = np.mean((recon - X_val[:100])**2)
print(f"\nMSE: {mse:.4f}")

## Cell 17 — Save & Download

In [ ]:
import shutil

# Save model
torch.save(model.state_dict(), "contrastive_ae_model.pth")

print("Files saved:")
for f in ["contrastive_ae_model.pth", "training_curves.png", "latent_umap.png", "clustering_analysis.png"]:
    print(f"  - {f}")

# Download in Colab/Kaggle
if IN_COLAB:
    from google.colab import files
    for f in ["contrastive_ae_model.pth", "training_curves.png", "latent_umap.png", "clustering_analysis.png"]:
        files.download(f)
elif IN_KAGGLE:
    print("\nIn Kaggle: Output files are in /kaggle/working/")

## Cell 18 — Summary

In [ ]:
print("""
=== CONTRASTIVE AUTOENCODER v2 - SUMMARY ===

Key differences from VAE v1:
- Removed KL divergence (was fighting against structure discovery)
- Added contrastive loss (pushes similar sequences together)
- Much smaller latent space (128 vs 1792)
- Trained on synthetic data with KNOWN structure first

Results to check:
1. Do discovered clusters match known patterns?
2. Is reconstruction better than v1?
3. Does contrastive loss create meaningful latent structure?

If this works on synthetic data, move to real genomic data.
""")